In [0]:
# Importar bibliotecas
from pyspark.sql.functions import col, month, year, sum as _sum, count, round, avg, to_date, expr, coalesce, lit, when

In [0]:
# 1. Leer los datsets crudos

df_nasa = spark.table("workspace.default.fire_archive")
df_inta = spark.table("workspace.default.temp_lluvias")
df_inta2 = spark.table("workspace.default.temp_lluvias2")

In [0]:
# 2. LIMPIEZA DE DATASET NASA (FUEGO)
# Filtramos la zona a Corrientes, Argentina y la confianza de la medición a mayor de 70
df_fuego_limpio = df_nasa.filter(
    (col("type") == 0) & 
    (col("latitude").between(-27.843933, -27.280264)) & 
    (col("longitude").between(-58.903198, -58.195953)) &
    (col("confidence").isin("n", "h") | expr("TRY_CAST(confidence AS INT) >= 70"))
)

# Extraemos Año y Mes
df_fuego_limpio = df_fuego_limpio.withColumn("Anio", year(col("acq_date"))) \
                                 .withColumn("Mes", month(col("acq_date")))

# AGRUPAMOS POR MES 
df_fuego_mensual = df_fuego_limpio.groupBy("Anio", "Mes").agg(
    count("*").alias("Total_Focos_Calor"),
    round(_sum("frp"), 2).alias("Intensidad_Total_FRP")
)

# Visualizamos la tabla MENSUAL ya procesada
display(df_fuego_mensual.limit(2))

In [0]:
# 3. LIMPIEZA Y REDUNDANCIA DE DATASET DEL INTA (CLIMA)

# Leer ambas tablas desde el Catalog (Dos sensores)
df_principal = spark.table("workspace.default.temp_lluvias")
df_respaldo = spark.table("workspace.default.temp_lluvias2")

# Limpiar tabla Principal (Sensor 1) 
df_p_clean = df_principal.select(
    to_date(col("Fecha")).alias("Fecha"),
    col("Precipitacion_Pluviometrica").cast("double").alias("Lluvia_P"),
    col("Temperatura_Abrigo_150cm_Maxima").cast("double").alias("Temp_P")
)

# Limpiar tabla de Respaldo (Sensor 2)
df_r_clean = df_respaldo.select(
    to_date(col("Fecha")).alias("Fecha"),
    col("Precipitacion_Pluviometrica").cast("double").alias("Lluvia_R"),
    col("Temperatura_Abrigo_150cm_Maxima").cast("double").alias("Temp_R")
)

# Crear el Calendario Maestro (Hay dias que faltan en el dataset original)
df_calendario = spark.sql("""
    SELECT explode(sequence(to_date('2016-06-01'), to_date('2026-03-29'), interval 1 day)) as Fecha
""")

# Unir Todo 
df_join = df_calendario.join(df_p_clean, on="Fecha", how="left") \
                       .join(df_r_clean, on="Fecha", how="left")

# Tratamiento de NULLS 
df_imputado = df_join.withColumn(
    "Lluvia_Real",
    coalesce(
        # Regla 1: Si Temp_P es null, el sensor falló. Forzamos un Null para que salte a la Regla 2.
        # Si no es null, usamos la lluvia principal.
        when(col("Temp_P").isNull(), lit(None)).otherwise(col("Lluvia_P")), 
        col("Lluvia_R"), # Regla 2: Usar el dato de la estación de respaldo
        lit(0.0)         # Regla 3: Si ambas estaciones explotaron, recién ahí ponemos 0.0
    )
).withColumn(
    "Temp_Real",
    # Para la temperatura, simplemente tomamos la principal, y si falla, la de respaldo
    coalesce(col("Temp_P"), col("Temp_R"))
)

# G. Agregar Año/Mes y Agrupar
df_clima_mensual = df_imputado.withColumn("Anio", year(col("Fecha"))) \
                              .withColumn("Mes", month(col("Fecha"))) \
                              .groupBy("Anio", "Mes").agg(
                                  round(_sum("Lluvia_Real"), 1).alias("Lluvia_Total_mm"),
                                  round(avg("Temp_Real"), 1).alias("Temp_Max_Mes")
                              ).orderBy("Anio", "Mes")

display(df_clima_mensual.limit(2))

In [0]:
# 4. FUSIÓN DE DATOS LIMPIOS (CLIMA + FUEGO)

# La tabla del Clima (que tiene todos los meses) como base en el LEFT JOIN.
df_analitica_final = df_clima_mensual.join(
    df_fuego_mensual, 
    on=["Anio", "Mes"], 
    how="left"
)

# 2. Imputación final: Los meses donde la NASA desapareció (nubes/falsos positivos)
# quedan en null tras el Join. Los forzamos a ser 0 para no romper las gráficas.
df_analitica_final = df_analitica_final.fillna({
    "Total_Focos_Calor": 0, 
    "Intensidad_Total_FRP": 0
})

# 3. Ordenamos todo cronológicamente para exportar
df_analitica_final = df_analitica_final.orderBy("Anio", "Mes")

# 4. Visualizamos la tabla lista para el Dashboard
display(df_analitica_final)

In [0]:
# 5. GUARDAR EN DATABRICKS (CAPA GOLD)
df_analitica_final.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.capstone_analitica_final")